In [1]:
# ── Cell 1: Imports & Configuration ──────────────────────────────────────────
import sys

import json
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import nibabel as nib
import pandas as pd
from skimage import measure

import ipywidgets as widgets
from IPython.display import display, clear_output

from utils.visualizer_utils import (
    hu_to_display, COLOR_AXIAL, COLOR_CORONAL, COLOR_SAGITTAL,
    load_decisions, save_decisions, overlay_multi_layer_mask,
    render_orthogonal_views, decision_badge, sync_sliders, sync_case_index,
    create_layer_checkboxes, create_overlay_control_panel, DEFAULT_LAYER_CONFIG,
    get_visible_labels, build_layer_status_suffix, create_viewer_layout,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_DIR    = Path('../data/dataset/Dataset820')
NIFTI_DIR      = DATASET_DIR / 'nifti'
MANIFEST_CSV   = DATASET_DIR / 'manifest.csv'
DECISIONS_FILE = DATASET_DIR / 'decisions.json'

# ── Protocol set (used for buttons) ──────────────────────────────────────────
PROTOCOLS = ['NC', 'VEN', 'ART', 'DELAY']

# ── HU display range ──────────────────────────────────────────────────────────
HU_MIN = -150
HU_MAX =  250

# ── Segmentation overlay configuration ────────────────────────────────────────
SEG_DIR        = DATASET_DIR / 'seg'
HAS_SEGMENTATION = SEG_DIR.exists()

# Multi-layer configuration (kidney=1, tumor=2, cyst=3)
LAYER_CONFIG = DEFAULT_LAYER_CONFIG.copy()
DEFAULT_VISIBLE = {1: True, 2: True, 3: False}  # Show kidney & tumor by default

# ── Sanity check ──────────────────────────────────────────────────────────────
assert NIFTI_DIR.exists(), f"nifti/ not found at {NIFTI_DIR}"
assert MANIFEST_CSV.exists(), f"manifest.csv not found at {MANIFEST_CSV}"

print("✓ Imports and configuration ready")
print(f"  DATASET_DIR    : {DATASET_DIR.resolve()}")
print(f"  NIFTI_DIR      : {NIFTI_DIR.resolve()}")
print(f"  MANIFEST_CSV   : {MANIFEST_CSV}")
print(f"  DECISIONS_FILE : {DECISIONS_FILE}")
print(f"  HU range       : [{HU_MIN}, {HU_MAX}] → normalised [0, 1]")
if HAS_SEGMENTATION:
    print(f"  ✓ Segmentation  : {SEG_DIR.resolve()}")
else:
    print(f"  ⚠ Segmentation  : folder not found (legacy dataset)")


✓ Imports and configuration ready
  DATASET_DIR    : /home/alonso/Documents/radio-ccrcc/data/dataset/Dataset820
  NIFTI_DIR      : /home/alonso/Documents/radio-ccrcc/data/dataset/Dataset820/nifti
  MANIFEST_CSV   : ../data/dataset/Dataset820/manifest.csv
  DECISIONS_FILE : ../data/dataset/Dataset820/decisions.json
  HU range       : [-150, 250] → normalised [0, 1]
  ✓ Segmentation  : /home/alonso/Documents/radio-ccrcc/data/dataset/Dataset820/seg


In [2]:
# ── Cell 2: Build cases from manifest & load existing decisions ───────────────

manifest_df = pd.read_csv(MANIFEST_CSV)

# ── Filter undefined phase ────────────────────────────────────────────────────
_undef_mask = manifest_df['phase'].str.strip().str.lower() == 'ven'
_undef_rows = manifest_df[_undef_mask]

all_cases = []
for _, row in _undef_rows.iterrows():
    filename = row['filename']
    all_cases.append({
        'filename': filename,
        'case_id':  row['case_id'],
        'group':    row['group'],
        'phase':    row['phase'],
        'filepath': NIFTI_DIR / filename,
        # Unique key (mirrors ct_phase_explorer key convention)
        'key':      f"{row['group']}/{row['case_id']}/{filename}",
    })

# ── Load decisions from disk ──────────────────────────────────────────────────
decisions = load_decisions(DECISIONS_FILE)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"✓ Manifest loaded    : {len(manifest_df)} total entries")
print(f"✓ Undefined phase    : {len(all_cases)} cases to review\n")
print(f"Phase distribution in manifest:")
print(manifest_df['phase'].value_counts().to_string())
print(f"\nGroup distribution in manifest:")
print(manifest_df['group'].value_counts().to_string())
print(f"\n✓ Loaded {len(decisions)} existing decisions from {DECISIONS_FILE.name}")
decided = defaultdict(int)
for e in decisions.values():
    decided[e['action']] += 1
for action, n in sorted(decided.items()):
    print(f"    {action}: {n}")


✓ Manifest loaded    : 1201 total entries
✓ Undefined phase    : 336 cases to review

Phase distribution in manifest:
phase
deleted    532
ven        336
nc         195
art        138

Group distribution in manifest:
group
NG    658
C     164
B     116
BD     75
A      71
AD     46
AC     27
BC     26
D      13
AB      5

✓ Loaded 1021 existing decisions from decisions.json
    ART: 103
    NC: 81
    VEN: 305
    delete: 532


In [3]:
# ── Cell 3: State Class ───────────────────────────────────────────────────────

class CaseReviewState:
    """
    Holds the full case list, current position, loaded arrays, and slice indices.

    Array convention (nibabel get_fdata): shape = (X, Y, Z)
      Axial    — image[:, :, az]   viewed along Z  → display as .T  (Y, X)
      Coronal  — image[:, cy, :]   viewed along Y  → display as .T  (Z, X)
      Sagittal — image[sx, :, :]   viewed along X  → display as .T  (Z, Y)
    """

    def __init__(self, cases: list, decisions: dict):
        self.cases     = cases
        self.decisions = decisions          # shared ref — mutations persist
        self.n_cases   = len(cases)
        self.idx       = 0                  # current case index

        # Loaded arrays
        self.image:   np.ndarray | None = None
        self.disp:    np.ndarray | None = None  # hu_to_display(image)
        self.seg:     np.ndarray | None = None  # segmentation mask (if present)
        self.spacing: tuple             = (1.0, 1.0, 1.0)  # (sx, sy, sz) mm

        # Current slice indices
        self.ax_idx:  int = 0   # axial   (Z axis)
        self.cor_idx: int = 0   # coronal (Y axis)
        self.sag_idx: int = 0   # sagittal (X axis)

    # ── Navigation ────────────────────────────────────────────────────────────

    def current_case(self) -> dict | None:
        if 0 <= self.idx < self.n_cases:
            return self.cases[self.idx]
        return None

    def go_next(self):
        if self.idx < self.n_cases - 1:
            self.idx += 1
            self.load_current()

    def go_prev(self):
        if self.idx > 0:
            self.idx -= 1
            self.load_current()
    
    def go_to_index(self, idx: int) -> bool:
        """Navigate to a specific case index. Returns True if successful."""
        if 0 <= idx < self.n_cases:
            self.idx = idx
            self.load_current()
            return True
        return False

    # ── Loading ───────────────────────────────────────────────────────────────

    def load_current(self) -> bool:
        """Load NIfTI volume for the current case; set slice indices to volume centre.
        
        Applies RAS reorientation for visualization only (original file unchanged).
        Attempts to load segmentation if folder exists.
        """
        case = self.current_case()
        if case is None:
            return False

        try:
            img = nib.load(str(case['filepath']))
            
            # Reorient to RAS for consistent visualization (non-destructive)
            img_ras = nib.as_closest_canonical(img)
            
            data       = img_ras.get_fdata()            # shape (X, Y, Z)
            self.image = data.astype(np.float32)
            self.disp  = hu_to_display(self.image, HU_MIN, HU_MAX)
            
            # Extract voxel spacing (sx, sy, sz) in mm from reoriented header
            zooms = img_ras.header.get_zooms()
            self.spacing = tuple(float(z) for z in zooms[:3])
            
            # ── Load segmentation if available ────────────────────────────────
            self.seg = None
            if HAS_SEGMENTATION:
                seg_filename = case['filename'].replace('_0000.nii.gz', '.nii.gz')
                seg_path = SEG_DIR / seg_filename
                if seg_path.exists():
                    try:
                        seg_img = nib.load(str(seg_path))
                        seg_ras = nib.as_closest_canonical(seg_img)
                        seg_data = seg_ras.get_fdata()
                        # Handle 4D volumes (time series) by taking first timepoint
                        if len(seg_data.shape) == 4:
                            seg_data = seg_data[..., 0]
                        self.seg = seg_data.astype(np.uint8)
                    except Exception as e:
                        pass  # silently skip malformed segmentation
            
            self._set_center_slices()
            return True

        except Exception as e:
            print(f"Error loading {case['filepath']}: {e}")
            self.image = self.disp = self.seg = None
            self.spacing = (1.0, 1.0, 1.0)
            return False

    def _set_center_slices(self):
        """Set ax/cor/sag to the geometric centre of the volume (no mask available)."""
        if self.image is None:
            self.sag_idx = self.cor_idx = self.ax_idx = 0
            return
        X, Y, Z      = self.image.shape
        self.sag_idx = X // 2
        self.cor_idx = Y // 2
        self.ax_idx  = Z // 2

    # ── Shapes / bounds ────────────────────────────────────────────────────────

    @property
    def shape(self):
        return self.image.shape if self.image is not None else (1, 1, 1)

    def max_ax(self):  return self.shape[2] - 1
    def max_cor(self): return self.shape[1] - 1
    def max_sag(self): return self.shape[0] - 1

    # ── Decision helpers ──────────────────────────────────────────────────────

    def get_decision(self) -> str | None:
        case = self.current_case()
        if case and case['key'] in self.decisions:
            return self.decisions[case['key']]['action']
        return None

    def set_decision(self, action: str):
        """Record a decision and immediately save to disk."""
        case = self.current_case()
        if case is None:
            return
        entry = {
            'key':       case['key'],
            'action':    action,
            'group':     case['group'],
            'case_id':   case['case_id'],
            'filename':  case['filename'],
            'old_phase': case['phase'],
        }
        self.decisions[case['key']] = entry
        save_decisions(self.decisions, DECISIONS_FILE)

    def clear_decision(self):
        """Remove decision for the current case."""
        case = self.current_case()
        if case and case['key'] in self.decisions:
            del self.decisions[case['key']]
            save_decisions(self.decisions, DECISIONS_FILE)


# ── Instantiate ───────────────────────────────────────────────────────────────
state = CaseReviewState(all_cases, decisions)
ok    = state.load_current()

if ok:
    c = state.current_case()
    sx, sy, sz = state.spacing
    print(f"✓ State ready  ({state.n_cases} cases)")
    print(f"  First case  : {c['key']}")
    print(f"  Array shape : {state.shape}  (X, Y, Z)")
    print(f"  Spacing     : {sx:.3f} × {sy:.3f} × {sz:.3f} mm")
    print(f"  Centre slice → axial={state.ax_idx}, coronal={state.cor_idx}, sagittal={state.sag_idx}")
    print(f"  NOTE: Images reoriented to RAS for visualization (original files unchanged)")
    if HAS_SEGMENTATION:
        seg_status = "✓ present" if state.seg is not None else "⚠ not found for this case"
        print(f"  Segmentation: {seg_status}")
else:
    print("⚠ No cases loaded — check manifest and NIFTI_DIR")


✓ State ready  (336 cases)
  First case  : BD/case_00153/00_case_00153_0000.nii.gz
  Array shape : (512, 512, 103)  (X, Y, Z)
  Spacing     : 0.939 × 0.939 × 5.000 mm
  Centre slice → axial=51, coronal=256, sagittal=256
  NOTE: Images reoriented to RAS for visualization (original files unchanged)
  Segmentation: ✓ present


In [4]:
# ── Cell 4: Render Function ───────────────────────────────────────────────────

def _overlay_segmentation_multi(ax_obj, seg_slice_2d: np.ndarray, visible_labels: dict):
    """Multi-layer overlay wrapper for nifti segmentation."""
    overlay_multi_layer_mask(ax_obj, seg_slice_2d, visible_labels, LAYER_CONFIG)


def render(output_widget: widgets.Output):
    """Render axial | coronal | sagittal into the given Output widget."""
    active_checkboxes = globals().get('layer_checkboxes', None)
    if active_checkboxes is None:
        visible_labels = DEFAULT_VISIBLE.copy()
    else:
        visible_labels = get_visible_labels(active_checkboxes)

    suffix = build_layer_status_suffix(
        visible_labels=visible_labels,
        layer_config=LAYER_CONFIG,
        has_overlay=(state.seg is not None),
    )

    render_orthogonal_views(
        state,
        output_widget,
        overlay_func=_overlay_segmentation_multi,
        case_title_suffix=suffix,
        visible_labels=visible_labels,
    )


In [5]:
# ── Cell 5: Widgets & Callbacks ───────────────────────────────────────────────

_updating = [False]   # guard flag in list so it can be modified by helper functions

# ── Output widget ─────────────────────────────────────────────────────────────
out = widgets.Output()

# ── Status label ──────────────────────────────────────────────────────────────
status_html = widgets.HTML()

# ── Layer control checkboxes ──────────────────────────────────────────────────
layer_checkboxes = create_layer_checkboxes(LAYER_CONFIG, DEFAULT_VISIBLE)
overlay_panel = create_overlay_control_panel(layer_checkboxes, LAYER_CONFIG)

# ── Case index text box ───────────────────────────────────────────────────────
txt_case_idx = widgets.BoundedIntText(
    value=1,
    min=1,
    max=len(all_cases),
    step=1,
    description='Case #:',
    layout=widgets.Layout(width='180px')
)

# ── Slice sliders (50% width, centered) ──────────────────────────────────────
_slider_layout = widgets.Layout(width='50%')

sl_ax  = widgets.IntSlider(description='Axial (Z)',    min=0, max=1, step=1,
                            continuous_update=True, layout=_slider_layout)
sl_cor = widgets.IntSlider(description='Coronal (Y)',  min=0, max=1, step=1,
                            continuous_update=True, layout=_slider_layout)
sl_sag = widgets.IntSlider(description='Sagittal (X)', min=0, max=1, step=1,
                            continuous_update=True, layout=_slider_layout)

# ── Buttons ───────────────────────────────────────────────────────────────────
btn_layout   = widgets.Layout(width='90px',  height='44px')
btn_layout_w = widgets.Layout(width='105px', height='44px')

btn_prev  = widgets.Button(description='← Prev',  button_style='info',    layout=btn_layout)
btn_next  = widgets.Button(description='Next →',  button_style='info',    layout=btn_layout)
btn_nc    = widgets.Button(description='NC',       button_style='success', layout=btn_layout)
btn_ven   = widgets.Button(description='VEN',      button_style='success', layout=btn_layout)
btn_art   = widgets.Button(description='ART',      button_style='success', layout=btn_layout)
btn_delay = widgets.Button(description='DELAY',    button_style='success', layout=btn_layout)
btn_skip  = widgets.Button(description='Skip',     button_style='warning', layout=btn_layout)
btn_del   = widgets.Button(description='✗ Delete', button_style='danger',  layout=btn_layout_w)

# ── Helpers ───────────────────────────────────────────────────────────────────

def _sync_sliders():
    """Push current state slice indices into sliders without triggering a render."""
    sync_sliders(state, sl_ax, sl_cor, sl_sag, _updating)

def _sync_case_index():
    """Update the case index text box without triggering a callback."""
    sync_case_index(state, txt_case_idx, _updating)

def _update_status():
    case     = state.current_case()
    decision = state.get_decision()
    if case is None:
        status_html.value = "<b style='color:green;font-size:14px;'>✓ All cases reviewed!</b>"
        return
    total_decided = len(state.decisions)
    sx, sy, sz    = state.spacing
    X, Y, Z       = state.shape
    
    # Segmentation status
    seg_status = "✓ present" if state.seg is not None else "✗ not found"
    seg_color = "#44cc44" if state.seg is not None else "#cc4444"
    
    status_html.value = f"""
    <div style='background:#1e1e1e;color:#ddd;padding:8px 12px;border-radius:6px;
                font-family:monospace;font-size:12px;line-height:1.7;'>
      <b style='font-size:13px;color:#fff;'>
        Case {state.idx + 1} / {state.n_cases}</b>
      &nbsp;·&nbsp; {decision_badge(decision)}<br/>
      <span style='color:#aaa;'>key:</span> {case['key']}<br/>
      <span style='color:#aaa;'>case_id:</span> {case['case_id']}
      &nbsp;·&nbsp;
      <span style='color:#aaa;'>group:</span> {case['group']}
      &nbsp;·&nbsp;
      <span style='color:#aaa;'>file:</span> {case['filename']}<br/>
      <span style='color:#aaa;'>phase:</span>
        <b style='color:#ffcc44;'>{case['phase'].upper()}</b>
      &nbsp;·&nbsp;
      <span style='color:#aaa;'>shape:</span> {X}×{Y}×{Z}
      &nbsp;·&nbsp;
      <span style='color:#aaa;'>spacing:</span>
        <b style='color:#88ccff;'>{sx:.3f} × {sy:.3f} × {sz:.3f} mm</b><br/>
      <span style='color:#aaa;'>segmentation:</span>
        <b style='color:{seg_color};'>{seg_status}</b>
      &nbsp;·&nbsp;
      <span style='color:#aaa;'>decisions:</span> {total_decided}
    </div>"""

def _full_refresh():
    """Sync sliders, update status bar, and re-render the figure."""
    _sync_sliders()
    _sync_case_index()
    _update_status()
    render(out)

# ── Slider callbacks ──────────────────────────────────────────────────────────

def _on_ax_change(change):
    if _updating[0]: return
    state.ax_idx = change['new']
    render(out)

def _on_cor_change(change):
    if _updating[0]: return
    state.cor_idx = change['new']
    render(out)

def _on_sag_change(change):
    if _updating[0]: return
    state.sag_idx = change['new']
    render(out)

sl_ax.observe(_on_ax_change,  names='value')
sl_cor.observe(_on_cor_change, names='value')
sl_sag.observe(_on_sag_change, names='value')

# ── Layer checkbox callbacks ──────────────────────────────────────────────────

def _on_layer_toggle(change):
    """Re-render when any layer checkbox is toggled."""
    if _updating[0]: return
    render(out)

for checkbox in layer_checkboxes.values():
    checkbox.observe(_on_layer_toggle, names='value')

# ── Case index text box callback ──────────────────────────────────────────────

def _on_case_idx_change(change):
    """Navigate to the specified case index (1-indexed)."""
    if _updating[0]: return
    target_idx = change['new'] - 1  # Convert to 0-indexed
    if state.go_to_index(target_idx):
        _full_refresh()
    else:
        # Invalid index, revert to current
        _sync_case_index()

txt_case_idx.observe(_on_case_idx_change, names='value')

# ── Button callbacks ──────────────────────────────────────────────────────────

def _classify(protocol: str):
    """Record a classify decision, advance, refresh."""
    state.set_decision(protocol)
    state.go_next()
    _full_refresh()

def _on_nc(_):    _classify('NC')
def _on_ven(_):   _classify('VEN')
def _on_art(_):   _classify('ART')
def _on_delay(_): _classify('DELAY')

def _on_delete(_):
    """Record a delete decision, advance, refresh."""
    state.set_decision('delete')
    state.go_next()
    _full_refresh()

def _on_skip(_):
    """Advance without recording any decision."""
    state.go_next()
    _full_refresh()

def _on_next(_):
    """Navigate forward (no decision recorded)."""
    state.go_next()
    _full_refresh()

def _on_prev(_):
    """Navigate backward (no decision recorded)."""
    state.go_prev()
    _full_refresh()

btn_nc.on_click(_on_nc)
btn_ven.on_click(_on_ven)
btn_art.on_click(_on_art)
btn_delay.on_click(_on_delay)
btn_del.on_click(_on_delete)
btn_skip.on_click(_on_skip)
btn_next.on_click(_on_next)
btn_prev.on_click(_on_prev)

# ── Initial sync ──────────────────────────────────────────────────────────────
_sync_sliders()
_sync_case_index()
_update_status()

print("✓ Widgets and callbacks ready — run cell 6 to display the UI")


✓ Widgets and callbacks ready — run cell 6 to display the UI


In [6]:
# ── Cell 6: Display UI ────────────────────────────────────────────────────────

# Navigation row with case index text box
_nav_row = widgets.HBox(
    [btn_prev, txt_case_idx, btn_next],
    layout=widgets.Layout(
        justify_content='center',
        gap='8px',
        padding='6px 0',
    )
)

_btn_row = widgets.HBox(
    [btn_nc, btn_ven, btn_art, btn_delay, btn_skip, btn_del],
    layout=widgets.Layout(
        justify_content='center',
        gap='6px',
        padding='6px 0',
    )
)

# Sliders at 50% width, centered horizontally
_sliders = widgets.VBox(
    [sl_ax, sl_cor, sl_sag],
    layout=widgets.Layout(
        align_items='center',
        width='100%',
        padding='4px 0',
    )
)

ui = create_viewer_layout(
    status_html=status_html,
    out=out,
    overlay_panel=overlay_panel,
    sliders=_sliders,
    nav_row=_nav_row,
    btn_row=_btn_row,
)

# Initial render
_full_refresh()
display(ui)


In [7]:

# ── Cell 7: Apply Reclassifications ──────────────────────────────────────────
#
# Reads decisions.json and updates the manifest CSV in-place for every entry
# whose action is NC, VEN, ART, or DELAY.
# No files are moved — Dataset820 uses a flat folder structure.
#
# This cell is independent of the viewer kernel state and reads decisions.json
# directly, so it can be run in a fresh session.

import json, shutil
from pathlib import Path
from collections import defaultdict
import pandas as pd

_DATASET_DIR    = Path('../data/dataset/Dataset820')
_MANIFEST_CSV   = _DATASET_DIR / 'manifest.csv'
_DECISIONS_FILE = _DATASET_DIR / 'decisions.json'

assert _DECISIONS_FILE.exists(), f"decisions.json not found at {_DECISIONS_FILE}"

with open(_DECISIONS_FILE, 'r') as _f:
    _all_decisions = json.load(_f)

_reclassify = [e for e in _all_decisions
               if e.get('action', '').upper() in ('NC', 'VEN', 'ART', 'DELAY')]

print(f"decisions.json  : {len(_all_decisions)} total entries")
print(f"to reclassify   : {len(_reclassify)}\n")

if not _reclassify:
    print("Nothing to reclassify.")
else:
    _df = pd.read_csv(_MANIFEST_CSV)
    _updated = []
    _skipped = []

    for entry in _reclassify:
        filename  = entry['filename']
        new_phase = entry['action'].lower()
        old_phase = entry.get('old_phase', 'undefined')

        _mask = _df['filename'] == filename
        if _mask.any():
            _df.loc[_mask, 'phase']           = new_phase
            _df.loc[_mask, 'protocol_source'] = 'manual'
            _updated.append(filename)
            print(f"  {filename}: {old_phase} → {new_phase}")
        else:
            _skipped.append(filename)
            print(f"  WARNING: {filename} not found in manifest — skipped")

    _df.to_csv(_MANIFEST_CSV, index=False)

    # Save reclassification log
    _log_path = _DATASET_DIR / 'reclassification_log.json'
    with open(_log_path, 'w') as _f:
        json.dump({
            'total_reclassified': len(_updated),
            'reclassifications':  _reclassify,
        }, _f, indent=2)

    print(f"\n✓ Updated manifest rows : {len(_updated)}")
    if _skipped:
        print(f"⚠ Skipped (missing)    : {len(_skipped)}")
    print(f"✓ Manifest saved       → {_MANIFEST_CSV}")
    print(f"✓ Reclassification log → {_log_path}")

    # Breakdown by new phase
    by_phase = defaultdict(int)
    for e in _reclassify:
        by_phase[e['action'].upper()] += 1
    for phase, n in sorted(by_phase.items()):
        print(f"    → {phase}: {n} cases")


decisions.json  : 1021 total entries
to reclassify   : 489

  03_case_00001_0000.nii.gz: undefined → ven
  04_case_00001_0000.nii.gz: undefined → nc
  08_case_00001_0000.nii.gz: undefined → art
  05_case_00002_0000.nii.gz: undefined → ven
  06_case_00002_0000.nii.gz: undefined → ven
  08_case_00002_0000.nii.gz: undefined → ven
  10_case_00002_0000.nii.gz: undefined → ven
  00_case_00153_0000.nii.gz: undefined → ven
  21_case_00002_0000.nii.gz: undefined → ven
  22_case_00002_0000.nii.gz: undefined → ven
  25_case_00002_0000.nii.gz: undefined → ven
  32_case_00002_0000.nii.gz: undefined → ven
  33_case_00002_0000.nii.gz: undefined → ven
  00_case_00254_0000.nii.gz: undefined → art
  00_case_00095_0000.nii.gz: undefined → art
  03_case_00095_0000.nii.gz: nc → nc
  02_case_00282_0000.nii.gz: undefined → ven
  04_case_00282_0000.nii.gz: undefined → ven
  07_case_00282_0000.nii.gz: undefined → nc
  08_case_00282_0000.nii.gz: undefined → art
  00_case_00295_0000.nii.gz: art → art
  02_case_0

In [8]:

# ── Cell 8: Execute Deletions (non-destructive) ───────────────────────────────
#
# Files marked with action == 'delete' are MOVED (never unlink'd) to:
#   DATASET_DIR/deleted/<filename>
# You can restore any file by moving it back to nifti/ manually.
# The manifest phase is updated to 'deleted'.
# A deletion log is saved to DATASET_DIR/deletion_log.json.

import json, shutil
from pathlib import Path
import pandas as pd

_DATASET_DIR    = Path('../data/dataset/Dataset820')
_NIFTI_DIR      = _DATASET_DIR / 'nifti'
_DELETED_ROOT   = _DATASET_DIR / 'deleted'
_MANIFEST_CSV   = _DATASET_DIR / 'manifest.csv'
_DECISIONS_FILE = _DATASET_DIR / 'decisions.json'

assert _DECISIONS_FILE.exists(), f"decisions.json not found at {_DECISIONS_FILE}"

with open(_DECISIONS_FILE, 'r') as _f:
    _all_decisions = json.load(_f)

_to_delete = [e for e in _all_decisions if e.get('action') == 'delete']

print(f"decisions.json   : {len(_all_decisions)} total entries")
print(f"marked delete    : {len(_to_delete)}\n")

if not _to_delete:
    print("Nothing to do.")
else:
    _DELETED_ROOT.mkdir(parents=True, exist_ok=True)
    _df = pd.read_csv(_MANIFEST_CSV)
    _moved = []
    _skipped_missing = []

    for entry in _to_delete:
        filename = entry['filename']
        src      = _NIFTI_DIR / filename
        dst      = _DELETED_ROOT / filename

        if dst.exists():
            print(f"  Already moved: {filename}")
            continue
        if not src.exists():
            _skipped_missing.append(filename)
            print(f"  WARNING: {filename} not found in nifti/ — skipped")
            continue

        shutil.move(str(src), str(dst))
        _moved.append({'from': f"nifti/{filename}", 'to': f"deleted/{filename}"})
        print(f"  Moved: {filename}")

        # Update manifest
        _mask = _df['filename'] == filename
        if _mask.any():
            _df.loc[_mask, 'phase']           = 'deleted'
            _df.loc[_mask, 'protocol_source'] = 'manual'

    _df.to_csv(_MANIFEST_CSV, index=False)

    # Save deletion log (append to existing if present)
    _log_path = _DATASET_DIR / 'deletion_log.json'
    _existing_moved = []
    if _log_path.exists():
        with open(_log_path, 'r') as _f:
            _existing_moved = json.load(_f).get('moved_files', [])

    with open(_log_path, 'w') as _f:
        json.dump({
            'total_entries': len(_to_delete),
            'moved_files':   _existing_moved + _moved,
        }, _f, indent=2)

    print(f"\n✓ Moved to deleted/    : {len(_moved)} file(s)")
    if _skipped_missing:
        print(f"⚠ Skipped (missing)   : {len(_skipped_missing)}")
    print(f"✓ Manifest saved       → {_MANIFEST_CSV}")
    print(f"✓ Deletion log         → {_log_path}")


decisions.json   : 1021 total entries
marked delete    : 532

  Already moved: 07_case_00001_0000.nii.gz
  Already moved: 04_case_00002_0000.nii.gz
  Already moved: 13_case_00002_0000.nii.gz
  Already moved: 15_case_00002_0000.nii.gz
  Already moved: 00_case_00248_0000.nii.gz
  Already moved: 00_case_00001_0000.nii.gz
  Already moved: 01_case_00001_0000.nii.gz
  Already moved: 02_case_00001_0000.nii.gz
  Already moved: 18_case_00002_0000.nii.gz
  Already moved: 19_case_00002_0000.nii.gz
  Already moved: 20_case_00002_0000.nii.gz
  Already moved: 23_case_00002_0000.nii.gz
  Already moved: 24_case_00002_0000.nii.gz
  Already moved: 26_case_00002_0000.nii.gz
  Already moved: 28_case_00002_0000.nii.gz
  Already moved: 29_case_00002_0000.nii.gz
  Already moved: 34_case_00002_0000.nii.gz
  Already moved: 01_case_00254_0000.nii.gz
  Already moved: 02_case_00095_0000.nii.gz
  Already moved: 00_case_00282_0000.nii.gz
  Already moved: 01_case_00282_0000.nii.gz
  Already moved: 03_case_00282_0000

In [9]:

# ── Cell 9: Session Summary ───────────────────────────────────────────────────

import json, pandas as pd
from pathlib import Path
from collections import defaultdict

_DATASET_DIR    = Path('../data/dataset/Dataset820')
_MANIFEST_CSV   = _DATASET_DIR / 'manifest.csv'
_DECISIONS_FILE = _DATASET_DIR / 'decisions.json'
_PROTOCOLS      = ['NC', 'VEN', 'ART', 'DELAY']

print(f"\n{'='*70}")
print("SESSION SUMMARY")
print(f"{'='*70}")

# Load decisions
_decisions = []
if _DECISIONS_FILE.exists():
    with open(_DECISIONS_FILE, 'r') as _f:
        _decisions = json.load(_f)

_reclass  = [e for e in _decisions if e.get('action', '').upper() in ('NC', 'VEN', 'ART', 'DELAY')]
_deleted  = [e for e in _decisions if e.get('action') == 'delete']
_undecided = [e for e in _decisions if e.get('action') not in ('NC', 'VEN', 'ART', 'DELAY', 'delete')]

print(f"\nTotal decisions saved : {len(_decisions)}")
print(f"  Reclassified        : {len(_reclass)}")
print(f"  Marked for deletion : {len(_deleted)}")

print(f"\nPhase distribution (reclassifications):")
_phase_counts = defaultdict(int)
for e in _reclass:
    _phase_counts[e['action'].upper()] += 1
for phase in _PROTOCOLS:
    print(f"  {phase:<8}: {_phase_counts.get(phase, 0)} cases")

# Load manifest for final distribution
if _MANIFEST_CSV.exists():
    _df = pd.read_csv(_MANIFEST_CSV)
    print(f"\nFinal phase distribution in manifest:")
    print(_df['phase'].value_counts().to_string())

print(f"\nFiles:")
print(f"  decisions.json     : {_DECISIONS_FILE}")
if (_DATASET_DIR / 'reclassification_log.json').exists():
    print(f"  reclassification   : {_DATASET_DIR / 'reclassification_log.json'}")
if (_DATASET_DIR / 'deletion_log.json').exists():
    print(f"  deletion log       : {_DATASET_DIR / 'deletion_log.json'}")
if (_DATASET_DIR / 'deleted').exists():
    print(f"  deleted files      : {_DATASET_DIR / 'deleted/'}")

print(f"\nNext steps:")
print(f"  1. Verify phase counts above")
print(f"  2. Review logs in {_DATASET_DIR}")
print(f"  3. Run downstream pipeline with updated manifest")



SESSION SUMMARY

Total decisions saved : 1021
  Reclassified        : 489
  Marked for deletion : 532

Phase distribution (reclassifications):
  NC      : 81 cases
  VEN     : 305 cases
  ART     : 103 cases
  DELAY   : 0 cases

Final phase distribution in manifest:
phase
deleted    532
ven        336
nc         195
art        138

Files:
  decisions.json     : ../data/dataset/Dataset820/decisions.json
  reclassification   : ../data/dataset/Dataset820/reclassification_log.json
  deletion log       : ../data/dataset/Dataset820/deletion_log.json
  deleted files      : ../data/dataset/Dataset820/deleted

Next steps:
  1. Verify phase counts above
  2. Review logs in ../data/dataset/Dataset820
  3. Run downstream pipeline with updated manifest
